In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore') # Ekranı kirleten uyarıları kapatalım

# 1. Veriyi Oku ve Temel Dönüşümleri Yap
df = pd.read_csv("../data/processed/01_final_merged_data.csv", sep=';', decimal=',')

def target_belirle(row):
    surec = str(row.get('Süreç', '')).strip()
    neden = str(row.get('İade Nedeni', '')).strip()
    if surec == 'İptal': return 2
    elif surec == 'İade' or (neden != 'nan' and neden != ''): return 1
    return 0

df['target'] = df.apply(target_belirle, axis=1)

def platform_birlestir(x):
    x_lower = str(x).lower()
    if 'admin' in x_lower: return 'Admin'
    elif any(k in x_lower for k in ['mobil', 'app', 'ios', 'android']): return 'Mobil'
    return 'Masaüstü'

df['Platform_Gruplu'] = df['Platform'].apply(platform_birlestir)

top_10_iller = df['İl (Teslimat)'].value_counts().nlargest(10).index
df['İl_Gruplu'] = df['İl (Teslimat)'].where(df['İl (Teslimat)'].isin(top_10_iller), 'Diğer')

# Tutar sütununu sayısala çevirme (güvenlik amaçlı)
df['Tutar'] = pd.to_numeric(df['Tutar'], errors='coerce').fillna(0)

# 2. Kategorik Matrisi Çıkar ve Tuzakları Temizle
kat_ozellikler = ['Platform_Gruplu', 'Kaynak', 'Ödeme Tipi', 'Ana Kategori', 'İl_Gruplu']
X_kat = pd.get_dummies(df[kat_ozellikler].fillna('Bilinmiyor'), drop_first=True)

# SADECE Ki-Kare'den geçen Anlamlı Özellikler (Çoklu bağlantı yaratan Havale ve Masaüstü elendi)
kalacak_kategorik = [
    'Ana Kategori_Ayakkabı', 'Ana Kategori_Çanta', 'Ana Kategori_Bilinmiyor', 'Ana Kategori_Diğer',
    'Ödeme Tipi_Kredi Kartı', 'Platform_Gruplu_Mobil',
    'İl_Gruplu_İzmir', 'İl_Gruplu_İstanbul', 'İl_Gruplu_Ankara',
    'Kaynak_facebook', 'Kaynak_ig'
]

secilen_kat_sutunlar = [col for col in kalacak_kategorik if col in X_kat.columns]
X_final = X_kat[secilen_kat_sutunlar].copy()

# Sayısal özelliği (Tutar) matrise altın dokunuş olarak ekle
X_final['Tutar'] = df['Tutar']
y = df['target']

# 3. Eğitim/Test Ayrımı ve Ölçeklendirme
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42, stratify=y)

# Tutar değerlerini scale et (Standartlaştır)
scaler = StandardScaler()
# DataLeakage (Veri sızıntısı) olmaması için sadece Train setine fit_transform yapıyoruz
X_train['Tutar'] = scaler.fit_transform(X_train[['Tutar']])
X_test['Tutar'] = scaler.transform(X_test[['Tutar']])

# 4. Hiperparametre Optimizasyonu (GridSearch)
print("XGBoost Modeli için en iyi parametreler aranıyor (Lütfen bekleyin, yaklaşık 1-2 dakika sürebilir)...\n")

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)

# Modelin kendi kendine deneyeceği kombinasyonlar
param_grid = {
    'max_depth': [3, 5, 7],          # Ağaç ne kadar derine insin? (Overfitting kontrolü)
    'learning_rate': [0.01, 0.1, 0.2], # Ne kadar hızlı öğrensin?
    'n_estimators': [50, 100, 200]     # Kaç farklı ağaç kursun?
}

# 5 Fold CV ile GridSearch'ü başlat
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"En İyi Parametreler: {grid_search.best_params_}")
print(f"En İyi Cross-Validation F1 Skoru: {grid_search.best_score_:.4f}\n")

# Seçilen Şampiyon Model ile Test Sonuçları
best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test)

print("--- Optimize Edilmiş XGBoost Test Seti Raporu ---")
print(classification_report(y_test, y_pred, target_names=["Teslim (0)", "İade (1)", "İptal (2)"]))

XGBoost Modeli için en iyi parametreler aranıyor (Lütfen bekleyin, yaklaşık 1-2 dakika sürebilir)...

En İyi Parametreler: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 50}
En İyi Cross-Validation F1 Skoru: 0.7974

--- Optimize Edilmiş XGBoost Test Seti Raporu ---
              precision    recall  f1-score   support

  Teslim (0)       0.92      0.98      0.95       516
    İade (1)       0.96      0.77      0.85        65
   İptal (2)       0.78      0.46      0.58        54

    accuracy                           0.92       635
   macro avg       0.89      0.74      0.80       635
weighted avg       0.91      0.92      0.91       635



In [7]:
from sklearn.metrics import f1_score

# Seçilen en iyi model ile hem Train hem Test tahminlerini alıyoruz
y_train_pred = best_xgb.predict(X_train)
y_test_pred = best_xgb.predict(X_test)

train_f1 = f1_score(y_train, y_train_pred, average='macro')
test_f1 = f1_score(y_test, y_test_pred, average='macro')

print("--- Kesin Overfitting (Ezber) Kontrolü ---")
print(f"Eğitim Seti (Train) F1: {train_f1:.4f}")
print(f"Test Seti (Test) F1: {test_f1:.4f}")
print(f"Fark: {abs(train_f1 - test_f1):.4f}")

if abs(train_f1 - test_f1) > 0.05:
    print("DURUM: Overfitting Riski!")
else:
    print("DURUM: Model Güvenli, Ezber Yok!")

--- Kesin Overfitting (Ezber) Kontrolü ---
Eğitim Seti (Train) F1: 0.8324
Test Seti (Test) F1: 0.7961
Fark: 0.0363
DURUM: Model Güvenli, Ezber Yok!


In [8]:
import joblib
import os

# 1. Kayıt için klasör oluştur (Eğer yoksa)
os.makedirs('../models', exist_ok=True)

# 2. Şampiyon Modeli Kaydet
joblib.dump(best_xgb, '../models/xgboost_return_model.pkl')

# 3. Scaler'ı Kaydet (Yeni gelen siparişlerin Tutar'ını aynı oranda ölçeklendirmek için şart)
joblib.dump(scaler, '../models/tutar_scaler.pkl')

# 4. Modelin beklediği sütun sırasını kaydet (Canlı ortamda hata almamak için hayati adım)
joblib.dump(X_train.columns.tolist(), '../models/model_columns.pkl')

print("Model, Scaler ve Sütun formatları 'models/' klasörüne başarıyla kaydedildi!")

Model, Scaler ve Sütun formatları 'models/' klasörüne başarıyla kaydedildi!
